# 03 — Prototipado de Modelos

Pruebas rápidas de los pipelines antes de formalizarlos en `src/models/`.

**Objetivos:**
- Verificar que los pipelines corren end-to-end
- Explorar resultados preliminares de CV
- Ajuste básico de hiperparámetros (tuning)
- Visualizar matriz de confusión e importancia de características

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data.load import load_raw
from src.data.clean import encode_binary_targets
from src.config import (
    REGRESSION_FEATURE_COLS, REGRESSION_TARGET_HEIGHT, REGRESSION_TARGET_WEIGHT,
    CLASSIFICATION_FEATURE_COLS, CLASSIFICATION_TARGET_DEFAULT,
    RANDOM_STATE, TEST_SIZE
)
from src.models.train import (
    build_regression_pipeline,
    build_classification_pipeline,
    cross_validate_model,
)
from src.models.evaluate import (
    regression_report,
    classification_report_extended,
    plot_confusion_matrix,
    plot_feature_importance,
)
from sklearn.model_selection import train_test_split

%matplotlib inline

df = load_raw('../data/SLV2013_Public_Use.csv')

## 1. Regresión — Prueba rápida (altura)

In [ ]:
available = [c for c in REGRESSION_FEATURE_COLS if c in df.columns]
subset = df[available + [REGRESSION_TARGET_HEIGHT]].dropna()
X, y = subset[available], subset[REGRESSION_TARGET_HEIGHT]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

for model_name in ['linear', 'random_forest']:
    pipeline = build_regression_pipeline(model_name)
    cv_scores = cross_validate_model(pipeline, X_train, y_train, scoring=['r2', 'neg_mean_absolute_error'])
    pipeline.fit(X_train, y_train)
    test_scores = regression_report(pipeline, X_test, y_test)
    print(f'{model_name}: CV R2={cv_scores["test_r2_mean"]:.3f}, Test={test_scores}')

## 2. Clasificación — Prueba rápida (sobrepeso)

In [ ]:
target_col = CLASSIFICATION_TARGET_DEFAULT
df_cls = encode_binary_targets(df, [target_col])
available = [c for c in CLASSIFICATION_FEATURE_COLS if c in df_cls.columns]
subset = df_cls[available + [target_col]].dropna()
X, y = subset[available], subset[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

for model_name in ['logistic', 'random_forest']:
    pipeline = build_classification_pipeline(model_name, use_smote=True)
    cv_scores = cross_validate_model(pipeline, X_train, y_train, scoring=['f1', 'roc_auc'])
    pipeline.fit(X_train, y_train)
    test_scores = classification_report_extended(pipeline, X_test, y_test, model_name)
    print(f'{model_name}: CV F1={cv_scores["test_f1_mean"]:.3f}, Test={test_scores}')

## 3. Matriz de confusión

In [ ]:
pipeline = build_classification_pipeline('random_forest', use_smote=True)
pipeline.fit(X_train, y_train)
fig = plot_confusion_matrix(pipeline, X_test, y_test, target_name=target_col, save=False)
plt.show()

## 4. Importancia de características

In [ ]:
fig = plot_feature_importance(
    pipeline, available, top_n=15,
    model_name='random_forest_prototype', save=False
)
plt.show()

## 5. Ajuste de hiperparámetros (RandomizedSearchCV)

In [ ]:
from src.models.tune import tune_pipeline, PARAM_GRIDS

pipeline = build_classification_pipeline('random_forest', use_smote=True)
param_grid = PARAM_GRIDS['classification_random_forest']

search = tune_pipeline(pipeline, param_grid, X_train, y_train, search_type='random', n_iter=10)
print('Best estimator:', search.best_params_)